In [2]:
import pandas as pd
from datetime import datetime

In [12]:
import os
print("Current working directory:", os.getcwd())
print("Contents of root:", os.listdir('.'))
print("Contents of Data:", os.listdir('Data') if os.path.exists('Data') else "Data folder missing")
print("Contents of Data/Raw:", os.listdir('Data/Raw') if os.path.exists('Data/Raw') else "Raw folder missing")

Current working directory: c:\Solar Prediction\Data
Contents of root: ['01_DataCleaning.ipynb', 'Cleaned', 'Data', 'Raw']
Contents of Data: ['Cleaned']
Contents of Data/Raw: Raw folder missing


In [21]:
# Load raw data for both plants (weather and generation)
df_weather1 = pd.read_csv('Plant_1_Weather_Sensor_Data.csv')
df_weather2 = pd.read_csv('Plant_2_Weather_Sensor_Data.csv')
df_gen1 = pd.read_csv('Plant_1_Generation_Data.csv')
df_gen2 = pd.read_csv('Plant_2_Generation_Data.csv')

In [22]:
# Standardize datetime columns
df_weather1['DATE_TIME'] = pd.to_datetime(df_weather1['DATE_TIME'])
df_weather2['DATE_TIME'] = pd.to_datetime(df_weather2['DATE_TIME'])
df_gen1['DATE_TIME'] = pd.to_datetime(df_gen1['DATE_TIME'], format='%d-%m-%Y %H:%M')
df_gen2['DATE_TIME'] = pd.to_datetime(df_gen2['DATE_TIME'])

In [23]:
# Aggregate generation data by DATE_TIME (sum AC_POWER across inverters)
df_gen1_agg = df_gen1.groupby('DATE_TIME')['AC_POWER'].sum().reset_index()
df_gen2_agg = df_gen2.groupby('DATE_TIME')['AC_POWER'].sum().reset_index()

In [24]:
# Merge weather and aggregated generation for each plant
df_merged1 = pd.merge(df_weather1, df_gen1_agg, on='DATE_TIME', how='left')
df_merged2 = pd.merge(df_weather2, df_gen2_agg, on='DATE_TIME', how='left')

In [25]:
# Add PLANT_ID column
df_merged1['PLANT_ID'] = 4135001
df_merged2['PLANT_ID'] = 4136001

In [26]:
# Combine data from both plants
df = pd.concat([df_merged1, df_merged2], ignore_index=True)

In [42]:
df.isnull().sum()

DATE_TIME               0
PLANT_ID                0
AMBIENT_TEMPERATURE     0
MODULE_TEMPERATURE      0
IRRADIATION             0
AC_POWER               25
dtype: int64

In [27]:
# Drop unnecessary columns (e.g., SOURCE_KEY)
df = df.drop(columns=['SOURCE_KEY'])

In [37]:
# Select features (adapted to available columns: datetime, PLANT_ID, AMBIENT_TEMPERATURE, MODULE_TEMPERATURE, IRRADIATION, AC_POWER)
features = ['DATE_TIME', 'PLANT_ID', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'AC_POWER']
df_clean = df[features].dropna()

In [38]:
# Rename DATE_TIME to datetime for consistency
df_clean = df_clean.rename(columns={'DATE_TIME': 'datetime'})

In [39]:
# Create datetime index
df_clean.set_index('datetime', inplace=True)

In [40]:
# Save cleaned data
df_clean.to_csv('data/clean_solar_data.csv')

In [41]:
# Confirm success
print('Data cleaned and saved successfully to data/clean_solar_data.csv')
print(df_clean.head())
print(f'Shape: {df_clean.shape}')

Data cleaned and saved successfully to data/clean_solar_data.csv
                     PLANT_ID  AMBIENT_TEMPERATURE  MODULE_TEMPERATURE  \
datetime                                                                 
2020-05-15 00:00:00   4135001            25.184316           22.857507   
2020-05-15 00:15:00   4135001            25.084589           22.761668   
2020-05-15 00:30:00   4135001            24.935753           22.592306   
2020-05-15 00:45:00   4135001            24.846130           22.360852   
2020-05-15 01:00:00   4135001            24.621525           22.165423   

                     IRRADIATION  AC_POWER  
datetime                                    
2020-05-15 00:00:00          0.0       0.0  
2020-05-15 00:15:00          0.0       0.0  
2020-05-15 00:30:00          0.0       0.0  
2020-05-15 00:45:00          0.0       0.0  
2020-05-15 01:00:00          0.0       0.0  
Shape: (6416, 5)
